# EDA — Análise Exploratória dos Indicadores Educacionais de Alagoas

Notebook de análise exploratória respondendo às 4 perguntas analíticas do projeto.

**Perguntas:**
1. Alagoas está convergindo ou divergindo da média nacional no IDEB desde 2005?
2. Quais municípios mais melhoraram e mais pioraram entre 2005 e 2023?
3. Qual item de infraestrutura tem maior correlação com o IDEB?
4. Gasto por aluno (FUNDEB) explica o desempenho?

**Input:** `data/processed/ideb_series_al.parquet`

In [66]:
import sys
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

sys.path.insert(0, str(Path().resolve().parent))
from src.config import IDEB_SERIES_PARQUET

# Médias nacionais do IDEB EF Anos Iniciais (fonte: INEP, divulgação Brasil)
IDEB_BRASIL = {
    2005: 3.8, 2007: 4.2, 2009: 4.6,
    2011: 4.9, 2013: 5.2, 2015: 5.3,
    2017: 5.5, 2019: 5.7, 2021: 5.8, 2023: 6.0
}

df = pd.read_parquet(IDEB_SERIES_PARQUET)
print(f"Dataset carregado: {df.shape}")
df.head(3)

Dataset carregado: (2371, 5)


,CO_MUNICIPIO,NO_MUNICIPIO,etapa,ano,ideb
0,2700102,Água Branca,EF Anos Iniciais,2005,2.4
1,2700102,Água Branca,EF Anos Iniciais,2007,2.8
2,2700102,Água Branca,EF Anos Iniciais,2009,3.1


## Pergunta 1 — Convergência ou divergência?
Alagoas está se aproximando ou se afastando da média nacional no IDEB desde 2005?

In [69]:
# Média do IDEB de AL por ano — apenas EF Anos Iniciais para comparar com Brasil
al_ai = (
    df[df["etapa"] == "EF Anos Iniciais"]
    .groupby("ano")["ideb"]
    .mean()
    .reset_index()
    .rename(columns={"ideb": "ideb_al"})
)

# Adicionar média nacional
al_ai["ideb_brasil"] = al_ai["ano"].map(IDEB_BRASIL)

# Gap entre AL e Brasil
al_ai["gap"] = (al_ai["ideb_brasil"] - al_ai["ideb_al"]).round(2)

print(al_ai.to_string(index=False))

 ano  ideb_al  ideb_brasil  gap
2005 2.426531          3.8 1.37
2007 2.910891          4.2 1.29
2009 3.290196          4.6 1.31
2011 3.326471          4.9 1.57
2013 3.590099          5.2 1.61
2015 4.224242          5.3 1.08
2017 4.786275          5.5 0.71
2019 5.269307          5.7 0.43
2021 5.324000          5.8 0.48
2023 5.910000          6.0 0.09


In [70]:
fig = go.Figure()

# Área do gap
fig.add_trace(go.Scatter(
    x=pd.concat([al_ai["ano"], al_ai["ano"][::-1]]),
    y=pd.concat([al_ai["ideb_brasil"], al_ai["ideb_al"][::-1]]),
    fill="toself", fillcolor="rgba(40, 163, 255, 0.36)",
    line=dict(color="rgba(0,0,0,0)"),
    name="Gap AL vs Brasil", hoverinfo="skip",
))

# Linha Brasil
fig.add_trace(go.Scatter(
    x=al_ai["ano"], y=al_ai["ideb_brasil"],
    name="Brasil", mode="lines+markers",
    line=dict(color="#378ADD", width=2),
    marker=dict(size=6),
    hovertemplate="Brasil %{x}: %{y:.2f}<extra></extra>",
))

# Linha Alagoas
fig.add_trace(go.Scatter(
    x=al_ai["ano"], y=al_ai["ideb_al"],
    name="Alagoas", mode="lines+markers",
    line=dict(color="#D85A30", width=2, dash="dash"),
    marker=dict(size=6),
    hovertemplate="Alagoas %{x}: %{y:.2f}<extra></extra>",
))

# Meta nacional
fig.add_hline(
    y=6.0, line_dash="dot", line_color="#888780",
    annotation_text="Meta 6.0",
    annotation_position="right",
)

# Anotações do gap nos anos mais relevantes
anos_destaque = {
    2005: "Gap: 1.37",
    2013: "Gap máx: 1.61",
    2019: "Gap: 0.43",
    2023: "Gap: 0.09",
}

for ano, texto in anos_destaque.items():
    linha = al_ai[al_ai["ano"] == ano].iloc[0]
    y_meio = (linha["ideb_brasil"] + linha["ideb_al"]) / 2.2
    fig.add_annotation(
        x=ano,
        y=y_meio,
        text=texto,
        showarrow=False,
        font=dict(size=9, color="#185FA5"),
        bgcolor="rgba(255,255,255,0.75)",
        bordercolor="#378ADD",
        borderwidth=0.5,
        borderpad=3,
    )

fig.update_layout(
    title="IDEB EF Anos Iniciais — Alagoas vs Brasil (2005–2023)",
    xaxis_title="Ano",
    yaxis_title="IDEB",
    yaxis=dict(range=[2.0, 7.0]),
    xaxis=dict(tickmode="array", tickvals=al_ai["ano"].tolist()),
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
    height=450,
)

fig.show()

### Interpretação

O gap entre Alagoas e Brasil **diminuiu consistentemente** entre 2005 e 2023,
passando de 1.37 para apenas 0.09 — a menor diferença em toda a série histórica.

**Período de maior aceleração:** entre 2015 e 2019, AL reduziu o gap em 0.65 pontos,
o avanço mais expressivo de qualquer quadriênio da série.

**Exceção à tendência:** entre 2009 e 2013 o gap aumentou de 1.31 para 1.61,
indicando que AL cresceu mais lentamente que a média nacional nesse período.

**Impacto da pandemia:** em 2021 o gap voltou a crescer levemente (0.43 → 0.48),
reflexo esperado do fechamento das escolas em 2020 — mas a recuperação em 2023
foi expressiva, com AL atingindo 5.91 e chegando a apenas 0.09 da meta nacional de 6.0.

**AL não atingiu a meta de 6.0** em nenhuma edição, mas 2023 representa o melhor
resultado histórico do estado e sugere que a meta pode ser alcançada na próxima edição.

## Pergunta 2 — Quais municípios mais melhoraram e mais pioraram?
Ranking dos municípios com maior variação do IDEB entre 2005 e 2023 (EF Anos Iniciais).

In [71]:
ai = df[df["etapa"] == "EF Anos Iniciais"].copy()

# Pegar IDEB de 2005 e 2023 para cada município
ideb_2005 = ai[ai["ano"] == 2005].set_index("CO_MUNICIPIO")["ideb"].rename("ideb_2005")
ideb_2023 = ai[ai["ano"] == 2023].set_index("CO_MUNICIPIO")["ideb"].rename("ideb_2023")
nomes     = ai[["CO_MUNICIPIO", "NO_MUNICIPIO"]].drop_duplicates().set_index("CO_MUNICIPIO")

variacao = (
    pd.concat([nomes, ideb_2005, ideb_2023], axis=1)
    .dropna(subset=["ideb_2005", "ideb_2023"])
    .assign(variacao=lambda x: (x["ideb_2023"] - x["ideb_2005"]).round(2))
    .sort_values("variacao", ascending=False)
    .reset_index()
)

print(f"Municípios com dados em 2005 e 2023: {len(variacao)}")
print(f"\nTop 5 que mais melhoraram:")
print(variacao.head(10)[["NO_MUNICIPIO","ideb_2005","ideb_2023","variacao"]].to_string(index=False))
print(f"\nTop 5 que mais pioraram:")
print(variacao.tail(10)[["NO_MUNICIPIO","ideb_2005","ideb_2023","variacao"]].to_string(index=False))

Municípios com dados em 2005 e 2023: 96

Top 5 que mais melhoraram:
      NO_MUNICIPIO  ideb_2005  ideb_2023  variacao
        Ibateguara        2.0        9.6       7.6
 Santana do Mundaú        2.7        9.8       7.1
União dos Palmares        2.6        9.7       7.1
          Coruripe        3.1        9.7       6.6
   Teotônio Vilela        2.6        9.0       6.4
   Jequiá da Praia        2.7        8.9       6.2
        Branquinha        2.3        8.3       6.0
         Junqueiro        2.8        8.4       5.6
  São José da Laje        3.0        8.3       5.3
      Campo Grande        1.8        7.0       5.2

Top 5 que mais pioraram:
           NO_MUNICIPIO  ideb_2005  ideb_2023  variacao
            Igreja Nova        2.5        5.0       2.5
São Miguel dos Milagres        2.4        4.8       2.4
                Craíbas        2.4        4.7       2.3
              Rio Largo        3.0        5.3       2.3
                 Maceió        3.1        5.3       2.2
         

In [73]:
print(f"Municípios com dados em 2005 E 2023 (no gráfico): {len(variacao)}")
print(f"Municípios com IDEB em 2005: {ai[ai['ano'] == 2005]['CO_MUNICIPIO'].nunique()}")
print(f"Municípios com IDEB em 2023: {ai[ai['ano'] == 2023]['CO_MUNICIPIO'].nunique()}")
print(f"Municípios excluídos do gráfico (sem dados em um dos anos): {102 - len(variacao)}")

Municípios com dados em 2005 E 2023 (no gráfico): 96
Municípios com IDEB em 2005: 98
Municípios com IDEB em 2023: 100
Municípios excluídos do gráfico (sem dados em um dos anos): 6


In [75]:
municipios_ausentes = [
    "2700904",  # Belo Monte
    "2703403",  # Jacaré dos Homens
    "2705309",  # Minador do Negrão
    "2705903",  # Olho d'Água Grande
    "2707602",  # Quebrangulo
    "2709004",  # Tanque d'Arca
]

for cod in municipios_ausentes:
    dados = df[
        (df["CO_MUNICIPIO"] == cod) &
        (df["etapa"] == "EF Anos Iniciais")
    ][["NO_MUNICIPIO", "ano", "ideb"]].sort_values("ano")
    
    nome = dados["NO_MUNICIPIO"].iloc[0]
    anos_disp = dados["ano"].tolist()
    print(f"\n{nome}")
    print(f"Anos disponíveis: {anos_disp}")
    print(dados[["ano", "ideb"]].to_string(index=False))


Belo Monte
Anos disponíveis: [2007, 2009, 2011, 2013, 2015, 2017, 2019, 2021, 2023]
 ano  ideb
2007   2.7
2009   4.2
2011   3.4
2013   3.2
2015   4.5
2017   3.8
2019   5.8
2021   5.3
2023   6.3

Jacaré dos Homens
Anos disponíveis: [2007, 2009, 2011, 2013, 2015, 2017, 2019, 2021, 2023]
 ano  ideb
2007   3.0
2009   3.8
2011   2.9
2013   3.9
2015   3.3
2017   4.8
2019   4.7
2021   5.5
2023   6.1

Minador do Negrão
Anos disponíveis: [2007, 2009, 2011, 2013, 2015, 2017, 2019, 2021, 2023]
 ano  ideb
2007   2.7
2009   2.9
2011   2.7
2013   3.2
2015   4.1
2017   4.3
2019   4.8
2021   4.7
2023   4.2

Olho d'Água Grande
Anos disponíveis: [2005, 2007, 2009, 2011, 2013, 2015, 2017, 2019, 2021]
 ano  ideb
2005   1.9
2007   2.4
2009   2.8
2011   3.4
2013   3.0
2015   3.6
2017   4.8
2019   4.3
2021   6.4

Quebrangulo
Anos disponíveis: [2007, 2009, 2011, 2013, 2015, 2017, 2019, 2021, 2023]
 ano  ideb
2007   2.8
2009   3.0
2011   3.6
2013   4.1
2015   5.0
2017   4.7
2019   5.7
2021   5.8
2023   5.8

T

In [76]:
# Municípios com IDEB em 2005
com_2005 = set(ai[ai["ano"] == 2005]["CO_MUNICIPIO"])

# Municípios com IDEB em 2023
com_2023 = set(ai[ai["ano"] == 2023]["CO_MUNICIPIO"])

# Todos os municípios de AL no dataset
todos = set(ai["CO_MUNICIPIO"].unique())

# Municípios que estão no gráfico (têm os dois anos)
no_grafico = set(variacao["CO_MUNICIPIO"])

# Ausentes do gráfico
ausentes = todos - no_grafico

# Montar tabela explicativa
nomes_todos = ai[["CO_MUNICIPIO", "NO_MUNICIPIO"]].drop_duplicates().set_index("CO_MUNICIPIO")

resultado = []
for cod in ausentes:
    nome = nomes_todos.loc[cod, "NO_MUNICIPIO"]
    tem_2005 = "✓" if cod in com_2005 else "✗"
    tem_2023 = "✓" if cod in com_2023 else "✗"
    resultado.append({
        "CO_MUNICIPIO": cod,
        "NO_MUNICIPIO": nome,
        "IDEB 2005": tem_2005,
        "IDEB 2023": tem_2023,
    })

df_ausentes = pd.DataFrame(resultado).sort_values("NO_MUNICIPIO")
print(f"Municípios ausentes do gráfico: {len(df_ausentes)}\n")
df_ausentes

Municípios ausentes do gráfico: 6



,CO_MUNICIPIO,NO_MUNICIPIO,IDEB 2005,IDEB 2023
2,2700904,Belo Monte,✗,✓
3,2703403,Jacaré dos Homens,✗,✓
5,2705309,Minador do Negrão,✗,✓
4,2705903,Olho d'Água Grande,✓,✗
1,2707602,Quebrangulo,✗,✓
0,2709004,Tanque d'Arca,✓,✗


#### Não serão incluídos nas análises os municípios de Belo Monte, Jacaré dos Homens, Minador do Negrão, Olho d'Água Grande, Quebrangulo, Tanque d'Arca por não possuirem um intervalo de daodos compatível com os demais municípios.

In [77]:
top10_melhores = variacao.head(10)
top10_piores   = variacao.tail(10).sort_values("variacao")

fig = go.Figure()

fig.add_trace(go.Bar(
    y=top10_melhores["NO_MUNICIPIO"],
    x=top10_melhores["variacao"],
    orientation="h", name="Mais melhoraram",
    marker_color="#1D9E75",
))

fig.add_trace(go.Bar(
    y=top10_piores["NO_MUNICIPIO"],
    x=top10_piores["variacao"],
    orientation="h", name="Mais pioraram",
    marker_color="#D85A30",
))

fig.update_layout(
    title="Top 10 municípios que mais melhoraram e mais pioraram no IDEB (2005–2023)",
    xaxis_title="Variação do IDEB",
    height=500,
    barmode="overlay",
)
fig.show()

In [78]:
fig = px.scatter(
    variacao,
    x="ideb_2005",
    y="variacao",
    #text="NO_MUNICIPIO",
    color="variacao",
    color_continuous_scale=["#D85A30", "#F1EFE8", "#1D9E75"],
    title="IDEB 2005 vs Variação 2005→2023 — Municípios de Alagoas",
    height=520,
    custom_data=["NO_MUNICIPIO", "ideb_2005", "ideb_2023", "variacao"],
)

fig.update_traces(
    marker=dict(size=8),
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "IDEB 2005: %{customdata[1]:.2f}<br>"
        "IDEB 2023: %{customdata[2]:.2f}<br>"
        "Variação:  %{customdata[3]:.2f}<br>"
        "<extra></extra>"
    ),
)

# Linha de referência: variação zero
fig.add_hline(
    y=0, line_dash="dash", line_color="#888780",
    annotation_text="Sem variação",
    annotation_position="right",
)

# Linha de referência: média AL em 2005
fig.add_vline(
    x=al_ai[al_ai["ano"] == 2005]["ideb_al"].values[0],
    line_dash="dash", line_color="#888780",
    annotation_text="Média AL 2005",
    annotation_position="top",
)

fig.update_traces(marker=dict(size=8))
fig.update_coloraxes(showscale=False)
fig.update_layout(showlegend=False)
fig.show()

### Interpretação

**Municípios que mais melhoraram (2005→2023):**
Os 10 municípios com maior variação positiva foram Ibateguara (+7.6),
Santana do Mundaú (+7.1), União dos Palmares (+7.1), Coruripe (+6.6),
Teotônio Vilela (+6.4), Jequiá da Praia (+6.2), Branquinha (+6.0),
Junqueiro (+5.6), São José da Laje (+5.3) e Campo Grande (+5.2).
Todos partiram de uma base baixa em 2005 (entre 1.8 e 3.1) e atingiram
os maiores IDEB do estado em 2023.

**Municípios que menos melhoraram:**
Nenhum município piorou entre 2005 e 2023 — todos tiveram variação positiva.
Os que menos avançaram foram Delmiro Gouveia (+1.6), Barra de Santo Antônio (+2.0),
Estrela de Alagoas (+2.0), Flexeiras (+2.0) e Traipu (+2.1).
Maceió, a capital, também figura entre os que menos melhoraram (+2.2),
partindo de 3.1 em 2005 e chegando a 5.3 em 2023.

**Padrão geográfico:**
Municípios menores do interior concentram os maiores avanços, enquanto
a região metropolitana (Maceió, Rio Largo) apresenta as menores variações.
Uma hipótese é que municípios com base muito baixa em 2005 tinham mais
espaço para crescer, além de possivelmente terem se beneficiado de
programas de alfabetização focados em municípios críticos.

**Nenhum município piorou desde 2005:**
O IDEB em 2005 variava entre 1.7 e 3.6 em Alagoas — uma faixa baixa
para todos os municípios. O município com maior IDEB em 2005 era Flexeiras
(3.6), que mesmo assim avançou para 5.6 em 2023.
